In [ ]:
!pip install transformers scikit-learn torch

In [ ]:
import json
import os
import numpy as np
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight

# Confirm GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME = "cardiffnlp/twitter-roberta-base-hate"
NUM_LABELS = 3
MAX_LENGTH = 128    # most HateXplain posts are short

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE    = 16
NUM_EPOCHS    = 5
LEARNING_RATE = 2e-5   # standard for fine-tuning RoBERTa
WARMUP_RATIO  = 0.1    # 10% of steps used for learning rate warmup
WEIGHT_DECAY  = 0.01

# ── Labels ─────────────────────────────────────────────────────────────────
LABEL_MAP   = {0: "hatespeech", 1: "offensive", 2: "normal"}
LABEL_NAMES = ["hatespeech", "offensive", "normal"]

Using device: cuda
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Update DATA_DIR if your files are in a different folder
DATA_DIR   = "/content/drive/MyDrive/hatexplain"
OUTPUT_DIR = "/content/drive/MyDrive/hatexplain/model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "hatexplain_train.jsonl")
VAL_PATH   = os.path.join(DATA_DIR, "hatexplain_validation.jsonl")
TEST_PATH  = os.path.join(DATA_DIR, "hatexplain_test.jsonl")

CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "best_model")

print(f"Data dir:   {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

Mounted at /content/drive
Data dir:   /content/drive/MyDrive/hatexplain
Output dir: /content/drive/MyDrive/hatexplain/model


In [ ]:
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def majority_vote(labels):
    return Counter(labels).most_common(1)[0][0]


def preprocess(data):
    sentences, labels = [], []
    for sample in data:
        sentences.append(" ".join(sample["post_tokens"]))
        labels.append(majority_vote(sample["annotators"]["label"]))
    return sentences, labels


train_data = read_jsonl(TRAIN_PATH)
val_data   = read_jsonl(VAL_PATH)
test_data  = read_jsonl(TEST_PATH)

train_sentences, train_labels = preprocess(train_data)
val_sentences,   val_labels   = preprocess(val_data)
test_sentences,  test_labels  = preprocess(test_data)

print(f"\nTrain: {len(train_sentences)} | Val: {len(val_sentences)} | Test: {len(test_sentences)}")

# Class distribution
for name, labels in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    c = Counter(labels)
    total = len(labels)
    dist = {LABEL_MAP[k]: f"{v} ({v/total*100:.1f}%)" for k, v in sorted(c.items())}
    print(f"{name}: {dist}")

# Compute class weights to handle imbalance at the loss level
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"\nClass weights: { {LABEL_MAP[i]: round(w, 3) for i, w in enumerate(class_weights)} }")



Train: 15383 | Val: 1922 | Test: 1924
Train: {'hatespeech': '4748 (30.9%)', 'offensive': '6251 (40.6%)', 'normal': '4384 (28.5%)'}
Val: {'hatespeech': '593 (30.9%)', 'offensive': '781 (40.6%)', 'normal': '548 (28.5%)'}
Test: {'hatespeech': '594 (30.9%)', 'offensive': '782 (40.6%)', 'normal': '548 (28.5%)'}

Class weights: {'hatespeech': np.float64(1.08), 'offensive': np.float64(0.82), 'normal': np.float64(1.17)}


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class HateXplainDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            sentences,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx]
        }


train_dataset = HateXplainDataset(train_sentences, train_labels, tokenizer, MAX_LENGTH)
val_dataset   = HateXplainDataset(val_sentences,   val_labels,   tokenizer, MAX_LENGTH)
test_dataset  = HateXplainDataset(test_sentences,  test_labels,  tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print(f"\nTrain batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]


Train batches: 962 | Val batches: 121 | Test batches: 121


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True  # replaces original classifier head with a fresh 3-class one
)
model = model.to(device)
print(f"\nModel loaded and moved to {device}")

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-hate
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
roberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.out_proj.bias        | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([2]) vs model:torch.Size([3])          
classifier.out_proj.weight      | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.



Model loaded and moved to cuda


In [ ]:
total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

print(f"\nTotal steps: {total_steps} | Warmup steps: {warmup_steps}")


def run_evaluation(loader):
    """Run model over a dataloader, return predictions and true labels."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_preds), np.array(all_labels)


best_val_f1 = 0.0
best_epoch  = 0

for epoch in range(NUM_EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()

        # Gradient clipping prevents exploding gradients during fine-tuning
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if (step + 1) % 100 == 0:
            print(f"  Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)

    # ── Validate ───────────────────────────────────────────────────────────
    val_preds, val_true = run_evaluation(val_loader)
    val_f1 = f1_score(val_true, val_preds, average="macro")

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} — Train loss: {avg_loss:.4f} | Val macro F1: {val_f1:.4f}")
    print(classification_report(val_true, val_preds, target_names=LABEL_NAMES))

    # ── Save best checkpoint ───────────────────────────────────────────────
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch  = epoch + 1
        model.save_pretrained(CHECKPOINT_PATH)
        tokenizer.save_pretrained(CHECKPOINT_PATH)
        print(f"  -> New best model saved to {CHECKPOINT_PATH} (macro F1: {best_val_f1:.4f})")

print(f"\nTraining complete. Best epoch: {best_epoch} | Best val macro F1: {best_val_f1:.4f}")



Total steps: 4810 | Warmup steps: 481
  Epoch 1 | Step 100/962 | Loss: 1.0962
  Epoch 1 | Step 200/962 | Loss: 0.8578
  Epoch 1 | Step 300/962 | Loss: 0.8593
  Epoch 1 | Step 400/962 | Loss: 0.6210
  Epoch 1 | Step 500/962 | Loss: 0.6981
  Epoch 1 | Step 600/962 | Loss: 0.7668
  Epoch 1 | Step 700/962 | Loss: 0.9927
  Epoch 1 | Step 800/962 | Loss: 0.9787
  Epoch 1 | Step 900/962 | Loss: 0.7789

Epoch 1/5 — Train loss: 0.8281 | Val macro F1: 0.6826
              precision    recall  f1-score   support

  hatespeech       0.76      0.79      0.78       593
   offensive       0.81      0.59      0.68       781
      normal       0.51      0.70      0.59       548

    accuracy                           0.68      1922
   macro avg       0.70      0.69      0.68      1922
weighted avg       0.71      0.68      0.68      1922



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> New best model saved to /content/drive/MyDrive/hatexplain/model/best_model (macro F1: 0.6826)
  Epoch 2 | Step 100/962 | Loss: 0.6694
  Epoch 2 | Step 200/962 | Loss: 0.3507
  Epoch 2 | Step 300/962 | Loss: 0.6794
  Epoch 2 | Step 400/962 | Loss: 0.5229
  Epoch 2 | Step 500/962 | Loss: 0.9026
  Epoch 2 | Step 600/962 | Loss: 0.7873
  Epoch 2 | Step 700/962 | Loss: 0.4942
  Epoch 2 | Step 800/962 | Loss: 0.6710
  Epoch 2 | Step 900/962 | Loss: 0.6671

Epoch 2/5 — Train loss: 0.6473 | Val macro F1: 0.6987
              precision    recall  f1-score   support

  hatespeech       0.75      0.82      0.79       593
   offensive       0.75      0.76      0.75       781
      normal       0.60      0.52      0.55       548

    accuracy                           0.71      1922
   macro avg       0.70      0.70      0.70      1922
weighted avg       0.71      0.71      0.71      1922



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> New best model saved to /content/drive/MyDrive/hatexplain/model/best_model (macro F1: 0.6987)
  Epoch 3 | Step 100/962 | Loss: 0.8308
  Epoch 3 | Step 200/962 | Loss: 0.5497
  Epoch 3 | Step 300/962 | Loss: 0.3347
  Epoch 3 | Step 400/962 | Loss: 0.3184
  Epoch 3 | Step 500/962 | Loss: 0.1774
  Epoch 3 | Step 600/962 | Loss: 0.6787
  Epoch 3 | Step 700/962 | Loss: 0.2533
  Epoch 3 | Step 800/962 | Loss: 0.5565
  Epoch 3 | Step 900/962 | Loss: 0.4794

Epoch 3/5 — Train loss: 0.5086 | Val macro F1: 0.6979
              precision    recall  f1-score   support

  hatespeech       0.77      0.82      0.79       593
   offensive       0.78      0.68      0.73       781
      normal       0.55      0.61      0.57       548

    accuracy                           0.70      1922
   macro avg       0.70      0.70      0.70      1922
weighted avg       0.71      0.70      0.70      1922

  Epoch 4 | Step 100/962 | Loss: 0.1378
  Epoch 4 | Step 200/962 | Loss: 0.2354
  Epoch 4 | Step 300/962 

In [ ]:
print("\nLoading best checkpoint for test evaluation...")
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_PATH)
model = model.to(device)

test_preds, test_true = run_evaluation(test_loader)
test_f1 = f1_score(test_true, test_preds, average="macro")

print("=" * 50)
print("Results on Test set (best checkpoint)")
print("=" * 50)
print(classification_report(test_true, test_preds, target_names=LABEL_NAMES))
print(f"Macro F1: {test_f1:.4f}")


Loading best checkpoint for test evaluation...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Results on Test set (best checkpoint)
              precision    recall  f1-score   support

  hatespeech       0.72      0.82      0.77       594
   offensive       0.74      0.77      0.76       782
      normal       0.58      0.45      0.51       548

    accuracy                           0.70      1924
   macro avg       0.68      0.68      0.68      1924
weighted avg       0.69      0.70      0.69      1924

Macro F1: 0.6774


In [ ]:
def predict(sentence: str) -> dict:
    """
    Classify a new sentence.
    Returns label, confidence, and full probability distribution.
    """
    model.eval()
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs   = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()

    pred_idx = int(np.argmax(probs))
    return {
        "label":         LABEL_MAP[pred_idx],
        "confidence":    round(float(probs[pred_idx]), 4),
        "probabilities": {LABEL_MAP[i]: round(float(p), 4) for i, p in enumerate(probs)}
    }


examples = [
    "I hate all people from that country, they should go back where they came from.",
    "This is a great movie, I really enjoyed it.",
    "These people are ruining our country and should be deported.",
    "Just tried the new bakery on 5th street. The croissants are amazing!",
    "I am so tired of this rain. It’s been gloomy for a straight week.",
    "You are literally the dumbest person on the internet. Delete your account",
    "Shut up and sit down. Your opinions are completely worthless.",
    "We need to get rid of every single Chinese in this neighborhood right now.",
    "If you are a Jew, you deserve to be beaten up. Simple as that.",
    "The new update for this app is completely broken. Fix it!!"
]

print("\nExample predictions:")
for sentence in examples:
    result = predict(sentence)
    print(f"\nInput      : \"{sentence}\"")
    print(f"Prediction : {result['label'].upper()} ({result['confidence']*100:.1f}% confidence)")
    print(f"Probs      : { {k: f'{v*100:.1f}%' for k, v in result['probabilities'].items()} }")


Example predictions:

Input      : "I hate all people from that country, they should go back where they came from."
Prediction : HATESPEECH (79.9% confidence)
Probs      : {'hatespeech': '79.9%', 'offensive': '5.1%', 'normal': '15.0%'}

Input      : "This is a great movie, I really enjoyed it."
Prediction : OFFENSIVE (92.8% confidence)
Probs      : {'hatespeech': '2.0%', 'offensive': '92.8%', 'normal': '5.2%'}

Input      : "These people are ruining our country and should be deported."
Prediction : HATESPEECH (69.7% confidence)
Probs      : {'hatespeech': '69.7%', 'offensive': '7.9%', 'normal': '22.4%'}

Input      : "Just tried the new bakery on 5th street. The croissants are amazing!"
Prediction : OFFENSIVE (92.0% confidence)
Probs      : {'hatespeech': '1.7%', 'offensive': '92.0%', 'normal': '6.2%'}

Input      : "I am so tired of this rain. It’s been gloomy for a straight week."
Prediction : OFFENSIVE (84.7% confidence)
Probs      : {'hatespeech': '4.0%', 'offensive': '84.7%', 'no